# 08. Agent / Priority Engine 전달용 Export

이 노트북은 06 위험도/리드타임 모델 결과와 07 근거 설명 결과를 합쳐서, 다음 단계의 Priority Engine 또는 Agent가 읽을 수 있는 입력 테이블을 만든다.

- 08에서는 모델을 새로 학습하지 않는다.
- 08의 산출물은 최종 점검 우선순위가 아니라, 우선순위 회귀/스코어링 모델에 넣을 입력값이다.
- hard label보다 `anomaly_score`, `risk_probability`, `lead_time_confidence`, `top_sensors`처럼 연속 신호와 근거를 최대한 보존한다.


In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)

RUN_ID = datetime.now().strftime("run_%Y%m%d_%H%M%S")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    candidates = [PROJECT_ROOT, *PROJECT_ROOT.parents]
    PROJECT_ROOT = next(path for path in candidates if (path / "data").exists())

SUPERVISED_DIR = PROJECT_ROOT / "data" / "processed" / "ml_supervised"
EXPLAINABILITY_DIR = PROJECT_ROOT / "data" / "processed" / "ml_explainability"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "ml_decision"
RUN_DIR = OUTPUT_DIR / "runs" / RUN_ID

for path in [OUTPUT_DIR, RUN_DIR]:
    path.mkdir(parents=True, exist_ok=True)

AGENT_OUTPUTS_PATH = SUPERVISED_DIR / "agent_model_outputs.csv"
EVIDENCE_PATH = EXPLAINABILITY_DIR / "decision_feature_evidence.csv"
OVER_RISK_GROUP_PATH = EXPLAINABILITY_DIR / "false_positive_group_diagnostics.csv"
SUPERVISED_META_PATH = SUPERVISED_DIR / "risk_leadtime_model_metadata.json"
EXPLAINABILITY_META_PATH = EXPLAINABILITY_DIR / "explainability_metadata.json"

required_paths = [AGENT_OUTPUTS_PATH, EVIDENCE_PATH, OVER_RISK_GROUP_PATH]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("08 실행 전 06/07 산출물이 필요합니다: " + ", ".join(missing_paths))

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"RUN_ID: {RUN_ID}")

## 1. 06/07 산출물 로드

06의 `agent_model_outputs.csv`를 기준 테이블로 사용하고, 07의 `decision_feature_evidence.csv`는 같은 window key에 근거 설명을 붙이는 보조 테이블로 사용한다.

In [ ]:
agent_outputs = pd.read_csv(AGENT_OUTPUTS_PATH)
evidence_raw = pd.read_csv(EVIDENCE_PATH)
over_risk_groups = pd.read_csv(OVER_RISK_GROUP_PATH)

with open(SUPERVISED_META_PATH, "r", encoding="utf-8") as fp:
    supervised_meta = json.load(fp)

with open(EXPLAINABILITY_META_PATH, "r", encoding="utf-8") as fp:
    explainability_meta = json.load(fp)

WINDOW_KEYS = ["manufacturer", "substation_id", "source_file", "window_start", "window_end"]

for frame_name, frame in [("agent_outputs", agent_outputs), ("evidence_raw", evidence_raw)]:
    missing_cols = [col for col in WINDOW_KEYS if col not in frame.columns]
    if missing_cols:
        raise KeyError(f"{frame_name}에 병합 key가 없습니다: {missing_cols}")

print("agent_outputs:", agent_outputs.shape)
print("evidence_raw:", evidence_raw.shape)
print("over_risk_groups:", over_risk_groups.shape)
display(agent_outputs.head(3))
display(evidence_raw.head(3))

## 2. 근거 테이블 정리 및 병합

07 근거는 고위험 또는 진단 대상 window 중심으로 생성되어 전체 window보다 행 수가 적다. 따라서 06 결과를 기준으로 left join하고, 근거가 없는 window는 빈 근거로 남긴다.

In [ ]:
def first_non_empty(series: pd.Series) -> str:
    valid = series.dropna().astype(str)
    valid = valid[valid.str.len() > 0]
    return valid.iloc[0] if len(valid) else ""


evidence_cols = [
    *WINDOW_KEYS,
    "top_sensors",
    "sensor_scores",
    "evidence_details",
    "pattern_notes",
    "explainability_run_id",
    "source_model_run_id",
    "priority_input_role",
    "diagnostic_label",
]
evidence_cols = [col for col in evidence_cols if col in evidence_raw.columns]

evidence = (
    evidence_raw[evidence_cols]
    .groupby(WINDOW_KEYS, as_index=False)
    .agg(first_non_empty)
)

decision = agent_outputs.merge(evidence, on=WINDOW_KEYS, how="left")

for col in ["top_sensors", "sensor_scores", "evidence_details", "pattern_notes", "priority_input_role", "diagnostic_label"]:
    if col not in decision.columns:
        decision[col] = ""
    decision[col] = decision[col].fillna("")

if "explainability_run_id" not in decision.columns:
    decision["explainability_run_id"] = explainability_meta.get("run_id", "")
else:
    decision["explainability_run_id"] = decision["explainability_run_id"].fillna(explainability_meta.get("run_id", ""))

if "source_model_run_id" not in decision.columns:
    decision["source_model_run_id"] = supervised_meta.get("run_id", "")
else:
    decision["source_model_run_id"] = decision["source_model_run_id"].fillna(supervised_meta.get("run_id", ""))

print("decision merged:", decision.shape)
print("근거가 붙은 window 수:", (decision["top_sensors"].astype(str).str.len() > 0).sum())
display(decision.head(3))

## 3. Priority Engine 입력 후보 신호 생성

여기서 만드는 `priority_input_score`와 `status_candidate`는 최종 운영 판단이 아니다. 다음 단계에서 회귀/스코어링 모델 또는 Agent가 여러 신호를 조합하기 쉽게 만든 baseline 입력값이다.

In [ ]:
numeric_cols = [
    "anomaly_score",
    "anomaly_threshold",
    "anomaly_label",
    "risk_probability",
    "risk_score",
    "risk_threshold",
    "risk_class",
    "lead_time_confidence",
    "leadtime_prob_short_0_24h",
    "leadtime_prob_mid_24_72h",
    "leadtime_prob_long_72h_plus",
]
for col in numeric_cols:
    if col in decision.columns:
        decision[col] = pd.to_numeric(decision[col], errors="coerce")

leadtime_urgency_map = {
    "short_0_24h": 1.00,
    "mid_24_72h": 0.65,
    "long_72h_plus": 0.35,
}

decision["leadtime_urgency_score"] = decision["lead_time_bucket"].map(leadtime_urgency_map).fillna(0.20)
decision["risk_margin"] = decision["risk_probability"] - decision["risk_threshold"]
decision["anomaly_margin"] = decision["anomaly_score"] - decision["anomaly_threshold"]
decision["has_evidence"] = decision["top_sensors"].astype(str).str.len() > 0

risk_probability = decision["risk_probability"].fillna(0).clip(0, 1)
lead_confidence = decision["lead_time_confidence"].fillna(0).clip(0, 1)
anomaly_signal = decision["anomaly_label"].fillna(0).clip(0, 1)
urgency_signal = decision["leadtime_urgency_score"].fillna(0).clip(0, 1)

decision["priority_input_score"] = (
    0.40 * risk_probability
    + 0.25 * urgency_signal
    + 0.20 * anomaly_signal
    + 0.15 * lead_confidence
).clip(0, 1)

def status_candidate(row: pd.Series) -> str:
    score = row.get("priority_input_score", 0)
    risk_prob = row.get("risk_probability", 0)
    risk_class = row.get("risk_class", 0)
    anomaly_label = row.get("anomaly_label", 0)
    lead_bucket = row.get("lead_time_bucket", "")

    if risk_class >= 1 and lead_bucket == "short_0_24h" and (anomaly_label >= 1 or risk_prob >= 0.98):
        return "긴급"
    if score >= 0.78 or risk_class >= 1:
        return "경고"
    if score >= 0.55 or risk_prob >= 0.80 or anomaly_label >= 1:
        return "주의"
    return "관찰"


decision["status_candidate"] = decision.apply(status_candidate, axis=1)
decision["priority_input_role"] = np.where(
    decision["priority_input_role"].astype(str).str.len() > 0,
    decision["priority_input_role"],
    "model_signal_without_local_evidence",
)

display(decision[["risk_probability", "risk_threshold", "risk_margin", "anomaly_score", "anomaly_margin", "lead_time_bucket", "leadtime_urgency_score", "priority_input_score", "status_candidate"]].head(10))

## 4. 과대위험 가능 group flag

07에서 확인한 holdout 과대위험 group을 이용해, Priority Engine이 특정 조건에서 risk 입력값을 보수적으로 해석할 수 있도록 진단 flag를 붙인다.

In [ ]:
diagnostic_groups = over_risk_groups.copy()
for col in ["false_positive", "false_positive_rate"]:
    diagnostic_groups[col] = pd.to_numeric(diagnostic_groups[col], errors="coerce")

diagnostic_groups = diagnostic_groups[
    (diagnostic_groups["false_positive"].fillna(0) > 0)
    & (diagnostic_groups["false_positive_rate"].fillna(0) >= 0.10)
]

diagnostic_map = {}
for _, row in diagnostic_groups.iterrows():
    diagnostic_map.setdefault(row["group_column"], set()).add(str(row["group_value"]))

def matched_diagnostic_groups(row: pd.Series) -> list[str]:
    matched = []
    for group_col, values in diagnostic_map.items():
        if group_col in row.index and str(row[group_col]) in values:
            matched.append(f"{group_col}={row[group_col]}")
    return matched


matched_groups = decision.apply(matched_diagnostic_groups, axis=1)
decision["overestimated_risk_group_flag"] = matched_groups.apply(lambda values: len(values) > 0)
decision["overestimated_risk_group_count"] = matched_groups.apply(len)
decision["overestimated_risk_group_notes"] = matched_groups.apply(lambda values: " | ".join(values))

display(diagnostic_groups[["group_column", "group_value", "false_positive", "false_positive_rate", "rows"]])
display(decision[decision["overestimated_risk_group_flag"]].head(5))

## 5. Export 테이블 구성

요약 테이블은 Agent나 Priority Engine이 빠르게 읽는 용도이고, 상세 테이블은 근거 JSON 문자열과 진단 정보를 보존하는 용도다.

In [ ]:
summary_cols = [
    "manufacturer",
    "substation_id",
    "source_file",
    "window_start",
    "window_end",
    "configuration_type",
    "season_bucket",
    "split_time_based",
    "split_regime_based",
    "anomaly_score",
    "anomaly_label",
    "risk_probability",
    "risk_threshold",
    "risk_class",
    "risk_level",
    "lead_time_bucket",
    "lead_time_confidence",
    "leadtime_urgency_score",
    "priority_input_score",
    "status_candidate",
    "top_sensors",
    "pattern_notes",
    "overestimated_risk_group_flag",
    "overestimated_risk_group_notes",
    "run_id",
    "explainability_run_id",
]
summary_cols = [col for col in summary_cols if col in decision.columns]

detail_cols = [
    *summary_cols,
    "sensor_scores",
    "evidence_details",
    "priority_input_role",
    "diagnostic_label",
    "risk_margin",
    "anomaly_margin",
    "leadtime_prob_short_0_24h",
    "leadtime_prob_mid_24_72h",
    "leadtime_prob_long_72h_plus",
    "risk_model_version",
    "leadtime_model_version",
    "source_model_run_id",
]
detail_cols = list(dict.fromkeys([col for col in detail_cols if col in decision.columns]))

priority_input_table = decision[summary_cols].sort_values(
    ["priority_input_score", "risk_probability", "lead_time_confidence"],
    ascending=[False, False, False],
).reset_index(drop=True)

agent_detail_export = decision[detail_cols].sort_values(
    ["priority_input_score", "risk_probability", "lead_time_confidence"],
    ascending=[False, False, False],
).reset_index(drop=True)

latest_by_substation = (
    decision.sort_values("window_end")
    .groupby(["manufacturer", "substation_id", "source_file"], as_index=False)
    .tail(1)
    .sort_values(["priority_input_score", "risk_probability"], ascending=[False, False])
    .reset_index(drop=True)
)
agent_summary_export = latest_by_substation[summary_cols]

display(priority_input_table.head(10))
display(agent_summary_export.head(10))

## 6. CSV / JSON 저장

실행별 run 폴더에 별도 파일을 남기고, 동시에 최신본 파일도 `data/processed/ml_decision/`에 덮어쓴다.

In [ ]:
def safe_json_loads(value: object, default: object) -> object:
    if not isinstance(value, str) or not value.strip():
        return default
    try:
        return json.loads(value)
    except json.JSONDecodeError:
        return default


def to_json_value(value: object) -> object:
    if pd.isna(value):
        return None
    if isinstance(value, np.generic):
        return value.item()
    return value


def make_agent_record(row: pd.Series) -> dict:
    return {
        "asset": {
            "manufacturer": row.get("manufacturer", ""),
            "substation_id": str(row.get("substation_id", "")),
            "source_file": row.get("source_file", ""),
            "configuration_type": row.get("configuration_type", ""),
        },
        "window": {
            "start": row.get("window_start", ""),
            "end": row.get("window_end", ""),
            "season_bucket": row.get("season_bucket", ""),
        },
        "signals": {
            "anomaly_score": to_json_value(row.get("anomaly_score", None)),
            "anomaly_label": to_json_value(row.get("anomaly_label", None)),
            "risk_probability": to_json_value(row.get("risk_probability", None)),
            "risk_level": row.get("risk_level", ""),
            "lead_time_bucket": row.get("lead_time_bucket", ""),
            "lead_time_confidence": to_json_value(row.get("lead_time_confidence", None)),
            "priority_input_score": to_json_value(row.get("priority_input_score", None)),
            "status_candidate": row.get("status_candidate", ""),
        },
        "evidence": {
            "top_sensors": safe_json_loads(row.get("top_sensors", ""), []),
            "sensor_scores": safe_json_loads(row.get("sensor_scores", ""), {}),
            "evidence_details": safe_json_loads(row.get("evidence_details", ""), []),
            "pattern_notes": row.get("pattern_notes", ""),
        },
        "diagnostics": {
            "overestimated_risk_group_flag": bool(row.get("overestimated_risk_group_flag", False)),
            "overestimated_risk_group_notes": row.get("overestimated_risk_group_notes", ""),
            "priority_input_role": row.get("priority_input_role", ""),
        },
        "metadata": {
            "supervised_run_id": row.get("run_id", ""),
            "explainability_run_id": row.get("explainability_run_id", ""),
            "export_run_id": RUN_ID,
        },
    }


agent_records = [make_agent_record(row) for _, row in agent_summary_export.iterrows()]

exports = {
    "decision_features.csv": decision,
    "priority_input_table.csv": priority_input_table,
    "agent_summary_export.csv": agent_summary_export,
    "agent_detail_export.csv": agent_detail_export,
}

for file_name, frame in exports.items():
    frame.to_csv(RUN_DIR / file_name, index=False, encoding="utf-8-sig")
    frame.to_csv(OUTPUT_DIR / file_name, index=False, encoding="utf-8-sig")

for base_dir in [RUN_DIR, OUTPUT_DIR]:
    with open(base_dir / "agent_records.json", "w", encoding="utf-8") as fp:
        json.dump(agent_records, fp, ensure_ascii=False, indent=2)

metadata = {
    "run_id": RUN_ID,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "purpose": "Priority Engine / Agent 입력용 decision feature export",
    "is_final_priority_model": False,
    "source_supervised_run_id": supervised_meta.get("run_id", ""),
    "source_explainability_run_id": explainability_meta.get("run_id", ""),
    "input_files": {
        "agent_model_outputs": str(AGENT_OUTPUTS_PATH),
        "decision_feature_evidence": str(EVIDENCE_PATH),
        "overestimated_risk_group_diagnostics": str(OVER_RISK_GROUP_PATH),
    },
    "output_files": list(exports.keys()) + ["agent_records.json", "export_metadata.json"],
    "rows": {
        "decision_features": int(len(decision)),
        "priority_input_table": int(len(priority_input_table)),
        "agent_summary_export": int(len(agent_summary_export)),
        "agent_detail_export": int(len(agent_detail_export)),
    },
    "status_candidate_counts": {str(key): int(value) for key, value in decision["status_candidate"].value_counts(dropna=False).items()},
    "overestimated_risk_group_flag_count": int(decision["overestimated_risk_group_flag"].sum()),
    "priority_input_score_rule": "0.40*risk_probability + 0.25*leadtime_urgency + 0.20*anomaly_label + 0.15*lead_time_confidence",
}

for base_dir in [RUN_DIR, OUTPUT_DIR]:
    with open(base_dir / "export_metadata.json", "w", encoding="utf-8") as fp:
        json.dump(metadata, fp, ensure_ascii=False, indent=2)

print("저장 완료")
print("RUN_DIR:", RUN_DIR)
for file_name in metadata["output_files"]:
    print("-", file_name)
display(pd.DataFrame([metadata["rows"]]))
display(decision["status_candidate"].value_counts().rename_axis("status_candidate").reset_index(name="rows"))